# AnyRAG (PDF → chunks → FAISS → local LLM)



In [ ]:
# 1. Install only what Colab doesn't already have (no transformers pinning!)
!pip -q install pymupdf sentence-transformers faiss-cpu accelerate
print("Dependencies ready.")

Dependencies ready.


In [ ]:
# 2. Configuration
import json, os, re, shutil, statistics, time
from pathlib import Path

# To keep everything between sessions, mount Drive and use e.g.
# BASE_DIR = Path("/content/drive/MyDrive/doctor_rag")
BASE_DIR = Path("/content/doctor_rag")
DATA_DIR = BASE_DIR / "data" / "processed"
INDEX_DIR = BASE_DIR / "data" / "vectorstore"
for d in (BASE_DIR, DATA_DIR, INDEX_DIR):
    d.mkdir(parents=True, exist_ok=True)

# Chunking
CHUNK_SIZE = 1000        # target characters per chunk
CHUNK_OVERLAP = 150      # characters carried into the next chunk
MIN_PAGE_CHARS = 100     # pages shorter than this are skipped

# Models
EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
RERANK_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"
LLM_NAME = "Qwen/Qwen2.5-3B-Instruct"   # fp16 fits a T4 (~7 GB), no bitsandbytes needed

REBUILD = False  # True = ignore cached chunks/index and rebuild everything

In [ ]:
# 3. Get the PDF (reuses an existing one, otherwise asks for an upload once)
existing = sorted(BASE_DIR.glob("*.pdf"))

if existing:
    PDF_PATH = existing[0]
else:
    from google.colab import files
    uploaded = files.upload()
    pdfs = [n for n in uploaded if n.lower().endswith(".pdf")]
    if not pdfs:
        raise ValueError("Please upload a valid PDF file.")
    PDF_PATH = BASE_DIR / pdfs[0]
    shutil.move(pdfs[0], PDF_PATH)

DOCUMENT_ID = re.sub(r"\W+", "_", PDF_PATH.stem).strip("_").lower()
print(f"PDF: {PDF_PATH.name} ({PDF_PATH.stat().st_size / 1024**2:.1f} MB)")

Saving anatomy+phys+vol2a.pdf to anatomy+phys+vol2a.pdf
PDF: anatomy+phys+vol2a.pdf (46.0 MB)


In [ ]:
# 4. Extract and clean text page by page
import pymupdf  # 'import fitz' is deprecated

CONTROL_RE = re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]")
INDEX_LINE_RE = re.compile(r",\s*\d+(?:,\s*\d+)*\s*$")
NOISE_LINE_PATTERNS = [
    re.compile(r"^This content is available for free at\b.*$", re.I),  # running footer
    re.compile(r"^CHAPTER\s+\d+\s*\|.*$"),                             # running header/footer
    re.compile(r"^\d{1,4}$"),                                           # bare page numbers
]


def clean_page_text(raw: str) -> str:
    """Strip control chars, running headers/footers and collapse whitespace."""
    text = CONTROL_RE.sub(" ", raw)
    lines = [ln.strip() for ln in text.splitlines()]
    lines = [ln for ln in lines if ln and not any(p.match(ln) for p in NOISE_LINE_PATTERNS)]
    text = " ".join(lines)
    return re.sub(r"\s+", " ", text).strip()


def skip_reason(raw: str, text: str):
    """Return why a page should not be indexed, or None if it is useful content."""
    if raw and len(CONTROL_RE.findall(raw)) / len(raw) > 0.02:
        return "garbled_encoding"
    if raw.count(". . .") >= 5:
        return "table_of_contents"
    lines = [ln for ln in raw.splitlines() if ln.strip()]
    if len(lines) >= 15 and sum(bool(INDEX_LINE_RE.search(ln)) for ln in lines) / len(lines) > 0.35:
        return "index"
    if "Creative Commons Attribution" in raw:
        return "license_page"
    if len(text) < MIN_PAGE_CHARS:
        return "too_short"
    return None


pages, skipped = [], {}
with pymupdf.open(PDF_PATH) as pdf:
    total_pages = len(pdf)
    for page_no, page in enumerate(pdf, start=1):
        raw = page.get_text("text")
        text = clean_page_text(raw)
        reason = skip_reason(raw, text)
        if reason:
            skipped[page_no] = reason
            continue
        pages.append({
            "document_id": DOCUMENT_ID,
            "source_file": PDF_PATH.name,
            "pdf_page": page_no,
            "text": text,
        })

(DATA_DIR / "cleaned_pages.json").write_text(
    json.dumps(pages, ensure_ascii=False, indent=2), encoding="utf-8"
)

reasons = {}
for r in skipped.values():
    reasons[r] = reasons.get(r, 0) + 1

print(f"Total pages: {total_pages}")
print(f"Kept pages: {len(pages)}")
print(f"Skipped pages: {len(skipped)} -> {reasons}")
print(f"Characters kept: {sum(len(p['text']) for p in pages):,}")

Total pages: 540
Kept pages: 503
Skipped pages: 37 -> {'garbled_encoding': 1, 'license_page': 1, 'too_short': 13, 'index': 22}
Characters kept: 1,495,773


In [ ]:
# 5. Sentence-aware chunking (keeps page numbers for citations)
SENTENCE_SPLIT = re.compile(r"(?<=[.!?])\s+(?=[A-Z0-9\"“(])")


def split_long(sentence: str, size: int):
    """Hard-split a sentence longer than `size` on word boundaries."""
    parts, cur = [], ""
    for word in sentence.split():
        if cur and len(cur) + 1 + len(word) > size:
            parts.append(cur)
            cur = word
        else:
            cur = f"{cur} {word}".strip()
    if cur:
        parts.append(cur)
    return parts


def chunk_text(text: str, size: int = CHUNK_SIZE, overlap: int = CHUNK_OVERLAP):
    sentences = []
    for s in SENTENCE_SPLIT.split(text):
        sentences.extend(split_long(s, size) if len(s) > size else [s])

    chunks, current, current_len = [], [], 0
    for s in sentences:
        if current and current_len + len(s) + 1 > size:
            chunks.append(" ".join(current))
            carry, n = [], 0                      # overlap = trailing whole sentences
            for prev in reversed(current):
                if n + len(prev) > overlap:
                    break
                carry.insert(0, prev)
                n += len(prev) + 1
            current, current_len = carry, n
        current.append(s)
        current_len += len(s) + 1
    if current:
        chunks.append(" ".join(current))
    return chunks


chunks = []
for page in pages:
    for piece in chunk_text(page["text"]):
        chunks.append({
            "chunk_id": f"chunk_{len(chunks) + 1:06d}",
            "document_id": page["document_id"],
            "source_file": page["source_file"],
            "pdf_page": page["pdf_page"],
            "text": piece,
            "char_count": len(piece),
        })

(DATA_DIR / "chunks.json").write_text(
    json.dumps(chunks, ensure_ascii=False, indent=2), encoding="utf-8"
)

lengths = [c["char_count"] for c in chunks]
print(f"Chunks: {len(chunks)}")
print(f"Avg / min / max length: {statistics.mean(lengths):.0f} / {min(lengths)} / {max(lengths)}")
print("\nSample:", chunks[len(chunks) // 2]["text"][:400])

Chunks: 1925
Avg / min / max length: 823 / 10 / 1141

Sample: Only during pregnancy do prolactin levels rise in response to prolactin-releasing hormone (PRH) from the hypothalamus. Intermediate Pituitary: Melanocyte-Stimulating Hormone The cells in the zone between the pituitary lobes secrete a hormone known as melanocyte-stimulating hormone (MSH) that is formed by cleavage of the pro-opiomelanocortin (POMC) precursor protein. Local production of MSH in the 


In [ ]:
# 6. Embeddings + FAISS index (cached on disk)
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

INDEX_PATH = INDEX_DIR / "anatomy.index"
META_PATH = INDEX_DIR / "chunks_metadata.json"

embed_model = SentenceTransformer(EMBED_MODEL)

use_cache = False
if INDEX_PATH.exists() and META_PATH.exists() and not REBUILD:
    index = faiss.read_index(str(INDEX_PATH))
    cached = json.loads(META_PATH.read_text(encoding="utf-8"))
    use_cache = index.ntotal == len(cached) == len(chunks)

if use_cache:
    print(f"Loaded cached index with {index.ntotal} vectors.")
else:
    embeddings = embed_model.encode(
        [c["text"] for c in chunks],
        batch_size=64,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    ).astype("float32")

    index = faiss.IndexFlatIP(embeddings.shape[1])  # inner product == cosine (normalized)
    index.add(embeddings)
    faiss.write_index(index, str(INDEX_PATH))
    META_PATH.write_text(json.dumps(chunks, ensure_ascii=False), encoding="utf-8")
    print(f"Indexed {index.ntotal} chunks, dim={embeddings.shape[1]}.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (5 outputs left out)

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/31 [00:00<?, ?it/s]

Indexed 1925 chunks, dim=384.


In [ ]:
# 7. Retrieval: dense search (top 20) -> cross-encoder rerank -> top 5
from sentence_transformers import CrossEncoder

reranker = CrossEncoder(RERANK_MODEL)


def retrieve_context(query: str, top_k: int = 5, fetch_k: int = 20, rerank: bool = True):
    q = embed_model.encode([query], convert_to_numpy=True, normalize_embeddings=True).astype("float32")
    scores, ids = index.search(q, max(fetch_k, top_k))

    candidates = [
        {**chunks[int(i)], "similarity": float(s)}
        for s, i in zip(scores[0], ids[0]) if i >= 0
    ]
    if rerank and candidates:
        rerank_scores = reranker.predict([(query, c["text"]) for c in candidates])
        for c, rs in zip(candidates, rerank_scores):
            c["rerank_score"] = float(rs)
        candidates.sort(key=lambda c: c["rerank_score"], reverse=True)
    return candidates[:top_k]


for r in retrieve_context("How does insulin regulate blood glucose?", top_k=3):
    print(f"p.{r['pdf_page']} | sim={r['similarity']:.2f} | rerank={r.get('rerank_score', 0):.2f}")
    print(r["text"][:250], "\n")

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

p.267 | sim=0.76 | rerank=7.67
Figure 17.19 Homeostatic Regulation of Blood Glucose Levels Blood glucose concentration is tightly maintained between 70 mg/dL and 110 mg/dL. If blood glucose concentration rises above this range, insulin is released, which stimulates body cells to r 

p.265 | sim=0.65 | rerank=4.69
Glucagon plays an important role in blood glucose regulation; low blood glucose levels stimulate its release. • The beta cell produces the hormone insulin and makes up approximately 75 percent of each islet. Elevated blood glucose levels stimulate th 

p.268 | sim=0.63 | rerank=4.27
Finally, insulin promotes triglyceride and protein synthesis. The secretion of insulin is regulated through a negative feedback mechanism. As blood glucose levels decrease, further insulin release is inhibited. The pancreatic hormones are summarized  



In [ ]:
# 8. Load the local LLM (fp16 on the T4; no 4-bit/bitsandbytes needed for a 3B model)
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

if not torch.cuda.is_available():
    raise RuntimeError("No GPU detected. Use Runtime > Change runtime type > T4 GPU.")

tokenizer = AutoTokenizer.from_pretrained(LLM_NAME)
try:
    llm = AutoModelForCausalLM.from_pretrained(LLM_NAME, dtype=torch.float16, device_map="auto")
except TypeError:  # older transformers versions use torch_dtype
    llm = AutoModelForCausalLM.from_pretrained(LLM_NAME, torch_dtype=torch.float16, device_map="auto")
llm.eval()

print("Loaded", LLM_NAME, "on", torch.cuda.get_device_name(0))

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Loaded Qwen/Qwen2.5-3B-Instruct on Tesla T4


In [ ]:
# 9. Answer generation with citations
SYSTEM_PROMPT = (
    "You are a study assistant for an Anatomy & Physiology textbook. "
    "Answer ONLY from the numbered context excerpts. Cite the sources you use like [Source 2]. "
    "If the excerpts do not contain the answer, say: "
    "\"The textbook excerpts I found don't cover this.\" Do not guess or add outside facts. "
    "Be clear and concise. This is educational material, not medical advice."
)


def build_messages(question: str, contexts):
    blocks = [
        f"[Source {i}] (PDF page {c['pdf_page']})\n{c['text']}"
        for i, c in enumerate(contexts, start=1)
    ]
    user = "Context:\n\n" + "\n\n".join(blocks) + f"\n\nQuestion: {question}\nAnswer:"
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user},
    ]


def answer_question(question: str, top_k: int = 5, max_new_tokens: int = 450):
    contexts = retrieve_context(question, top_k=top_k)
    prompt = tokenizer.apply_chat_template(
        build_messages(question, contexts), tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(llm.device)

    with torch.inference_mode():
        output = llm.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=None, top_p=None, top_k=None,
            repetition_penalty=1.05,
            pad_token_id=tokenizer.eos_token_id,
        )
    text = tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    return {"question": question, "answer": text, "sources": contexts}


def ask(question: str, show_sources: bool = True):
    start = time.time()
    result = answer_question(question)
    print(f"Q: {question}\n\nA: {result['answer']}\n")
    if show_sources:
        pages_used = ", ".join(f"[{i}] p.{c['pdf_page']}" for i, c in enumerate(result["sources"], 1))
        print(f"Sources: {pages_used}")
    print(f"({time.time() - start:.1f}s)")
    return result

In [ ]:
# 10. Try it
_ = ask("How does insulin help regulate blood glucose levels?")

Q: How does insulin help regulate blood glucose levels?

A: Insulin helps regulate blood glucose levels by facilitating the uptake of glucose into body cells. It primarily targets skeletal muscle cells and adipose cells, which are the primary targets of insulin. When blood glucose levels rise, insulin is released, stimulating these cells to take glucose from the bloodstream. Conversely, when blood glucose levels drop, glucagon is released instead. Additionally, insulin promotes triglyceride and protein synthesis. Its secretion is regulated through a negative feedback mechanism; as blood glucose levels decrease, further insulin release is inhibited.

Sources: [1] p.267, [2] p.268, [3] p.265, [4] p.268, [5] p.268
(6.2s)


In [ ]:
_ = ask("What is the difference between the pituitary's anterior and posterior lobes?")

Q: What is the difference between the pituitary's anterior and posterior lobes?

A: The anterior pituitary lobe arises from the oral ectoderm and produces and secretes six hormones: growth hormone (GH), thyroid-stimulating hormone (TSH), adrenocorticotropic hormone (ACTH), follicle-stimulating hormone (FSH), luteinizing hormone (LH), and prolactin (PRL). It is connected to the hypothalamus by vasculature in the infundibulum. The posterior pituitary lobe arises from the neural ectoderm at the base of the hypothalamus and consists of neuroendocrine cells that store and release hormones produced in the hypothalamus: antidiuretic hormone (ADH) and oxytocin.

Sources: [1] p.244, [2] p.245, [3] p.68, [4] p.272, [5] p.278
(7.8s)
